# PlotProof — Acoustic Chainsaw Detection (real ML, honest metrics)

Trains the classifier behind PlotProof's acoustic monitoring: **chainsaw / heavy_vehicle / other**,
the exact `EventClass` values the app's `/api/acoustic/ingest` endpoint accepts.

**Why this exists:** the satellite deforestation U-Net was cut because it could not be trained
honestly. This model *can*: it uses a real public benchmark (ESC-50), the dataset's own
predefined 5-fold cross-validation protocol, and every number published to the app is a real
cross-validated measurement. Nothing here is simulated.

**What you get at the end**
- `plotproof_acoustic.tflite` — end-to-end model (raw 16 kHz waveform → class probabilities) for the gateway
- `acoustic-metrics.json` — the model card the PlotProof acoustic page renders (copy to `public/models/`)
- a SavedModel, for anyone who wants to keep training

**Runtime:** ~20–30 min on CPU, ~15 with a GPU runtime (Runtime → Change runtime type → T4).
The one slow step (embedding extraction) is cached to disk, so re-runs are fast.

**License note, up front:** ESC-50 is CC BY-NC 3.0 — this exact model is for research/demo.
For commercial deployment, retrain on field recordings or a permissively licensed set (e.g. FSD50K).


In [ ]:
# Setup. Colab ships TF and librosa; the rest is tiny.
%pip -q install tensorflow_hub soundfile
import json, os, time, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
import librosa

SEED = 17
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TF", tf.__version__)


In [ ]:
# ESC-50: 2,000 five-second environmental clips, 50 classes, 5 predefined CV folds.
# https://github.com/karolpiczak/ESC-50  (dataset license: CC BY-NC 3.0)
if not os.path.exists("ESC-50-master"):
    print("downloading ESC-50 (~600 MB)...")
    urllib.request.urlretrieve("https://github.com/karolpiczak/ESC-50/archive/master.zip", "esc50.zip")
    with zipfile.ZipFile("esc50.zip") as z:
        z.extractall(".")
meta = pd.read_csv("ESC-50-master/meta/esc50.csv")

# Map ESC-50's 50 categories onto the app's three event classes
# (lib/acoustic/types.ts: chainsaw | heavy_vehicle | other).
# ESC-50 has no logging-truck class, so heavy_vehicle is an ENGINE/MACHINERY
# PROXY (engine, helicopter, airplane, train). Replace with field audio later.
CLASSES = ["chainsaw", "heavy_vehicle", "other"]
VEHICLE_PROXY = {"engine", "helicopter", "airplane", "train"}

def to_class(category):
    if category == "chainsaw":
        return 0
    if category in VEHICLE_PROXY:
        return 1
    return 2

meta["y"] = meta["category"].map(to_class)
print(meta.groupby("y").size().rename(index=dict(enumerate(CLASSES))))
meta.head()


In [ ]:
# What the model hears: log-mel spectrograms, one clip per class of interest.
examples = [meta[meta.category == "chainsaw"].iloc[0],
            meta[meta.category == "engine"].iloc[0],
            meta[meta.category == "rain"].iloc[0]]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, row in zip(axes, examples):
    wav, sr = librosa.load("ESC-50-master/audio/" + row.filename, sr=16000, mono=True)
    m = librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=64)
    ax.imshow(librosa.power_to_db(m, ref=np.max), aspect="auto", origin="lower", cmap="magma")
    ax.set_title(row.category)
    ax.set_xlabel("frames"); ax.set_ylabel("mel bins")
plt.tight_layout(); plt.show()


In [ ]:
# YAMNet (AudioSet-pretrained, frozen) as the feature extractor. Per clip we
# pool its per-frame 1024-d embeddings with mean AND max -> one 2048-d vector.
# Cached to disk so re-running the notebook skips the slow extraction.
yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

CACHE = "esc50_yamnet_embeddings.npz"
if os.path.exists(CACHE):
    d = np.load(CACHE)
    X, y, folds = d["X"], d["y"], d["folds"]
else:
    feats = []
    t0 = time.time()
    for i, row in enumerate(meta.itertuples()):
        wav, _ = librosa.load("ESC-50-master/audio/" + row.filename, sr=16000, mono=True)
        _, emb, _ = yamnet(wav.astype(np.float32))
        emb = emb.numpy()
        feats.append(np.concatenate([emb.mean(0), emb.max(0)]))
        if (i + 1) % 200 == 0:
            print(f"{i + 1}/2000  ({time.time() - t0:.0f}s)")
    X = np.stack(feats).astype(np.float32)
    y = meta["y"].to_numpy()
    folds = meta["fold"].to_numpy()
    np.savez_compressed(CACHE, X=X, y=y, folds=folds)
print(X.shape, y.shape)


In [ ]:
# Honest evaluation: leave-one-fold-out over ESC-50's five PREDEFINED folds
# (clips from the same source recording never straddle train and test - a random
# split would flatter the score).
from sklearn.metrics import (accuracy_score, average_precision_score,
                             classification_report, confusion_matrix,
                             precision_recall_curve, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

def build_head():
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2048,)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(3, activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def fit_head(m, Xtr, ytr):
    cw = compute_class_weight("balanced", classes=np.arange(3), y=ytr)
    m.fit(Xtr, ytr, epochs=80, batch_size=64, verbose=0,
          validation_split=0.15, class_weight=dict(enumerate(cw)),
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    return m

fold_acc, all_true, all_prob = [], [], []
for f in sorted(np.unique(folds)):
    tr, te = folds != f, folds == f
    scaler = StandardScaler().fit(X[tr])
    m = fit_head(build_head(), scaler.transform(X[tr]), y[tr])
    prob = m.predict(scaler.transform(X[te]), verbose=0)
    fold_acc.append(accuracy_score(y[te], prob.argmax(1)))
    all_true.append(y[te]); all_prob.append(prob)
    print(f"fold {f}: accuracy {fold_acc[-1]:.3f}")

y_true = np.concatenate(all_true); y_prob = np.concatenate(all_prob)
y_pred = y_prob.argmax(1)
print(f"\nmean accuracy {np.mean(fold_acc):.3f} +/- {np.std(fold_acc):.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(cmap="Blues")
plt.show()


In [ ]:
# Operating point. A false chainsaw alarm erodes a farmer's trust fast, so we
# deploy at the smallest threshold giving >= 95% cross-validated precision, and
# report the recall that buys us. This threshold ships in the model card.
chain_true = (y_true == 0).astype(int)
chain_prob = y_prob[:, 0]
ap = average_precision_score(chain_true, chain_prob)
prec, rec, thr = precision_recall_curve(chain_true, chain_prob)
ok = np.where(prec[:-1] >= 0.95)[0]
if len(ok):
    i = ok[0]
else:
    i = int(np.argmax(prec[:-1]))
    print("95% precision not reachable on CV; using the best available point")
THRESHOLD, p_at, r_at = float(thr[i]), float(prec[i]), float(rec[i])
print(f"chainsaw average precision {ap:.3f}")
print(f"deploy threshold {THRESHOLD:.3f} -> precision {p_at:.3f}, recall {r_at:.3f}")
plt.plot(rec, prec)
plt.scatter([r_at], [p_at], color="red", zorder=3)
plt.xlabel("recall"); plt.ylabel("precision")
plt.title("chainsaw precision-recall (5-fold CV)"); plt.grid(alpha=0.3); plt.show()


## Reading the numbers honestly

- The accuracy above is measured on **clean, single-source, 5-second clips**. A sensor 200 m
  from a chainsaw, in rain, behind foliage, hears something much harder. Until the model is
  validated on field recordings, the model card carries that caveat verbatim.
- `heavy_vehicle` is a **proxy** built from ESC-50's engine/helicopter/airplane/train classes,
  because no logging-truck class exists in the dataset.
- The 95%-precision threshold trades recall for trust: some chainsaws are missed, near-zero
  false alarms. For a corroborating (not primary) evidence channel, that is the right trade.


In [ ]:
# Final model: same recipe, trained on all 2000 clips, then packaged END-TO-END
# (raw 16 kHz waveform in -> class probabilities out) using plain TF ops so the
# export has no Keras/hub loading dependencies.
scaler = StandardScaler().fit(X)
final_head = fit_head(build_head(), scaler.transform(X), y)
d1, d2 = [l for l in final_head.layers if isinstance(l, tf.keras.layers.Dense)]
W1, b1 = d1.get_weights(); W2, b2 = d2.get_weights()

class PlotProofAcoustic(tf.Module):
    def __init__(self, yamnet):
        super().__init__()
        self.yamnet = yamnet
        self.mu = tf.constant(scaler.mean_, tf.float32)
        self.sd = tf.constant(scaler.scale_, tf.float32)
        self.W1 = tf.constant(W1); self.b1 = tf.constant(b1)
        self.W2 = tf.constant(W2); self.b2 = tf.constant(b2)

    @tf.function(input_signature=[tf.TensorSpec([None], tf.float32, name="waveform_16khz")])
    def __call__(self, waveform):
        _, emb, _ = self.yamnet(waveform)
        pooled = tf.concat([tf.reduce_mean(emb, 0), tf.reduce_max(emb, 0)], 0)
        x = ((pooled - self.mu) / self.sd)[None, :]
        h = tf.nn.relu(x @ self.W1 + self.b1)
        probs = tf.nn.softmax(h @ self.W2 + self.b2)[0]
        return {"probs": probs}  # order: chainsaw, heavy_vehicle, other

module = PlotProofAcoustic(yamnet)
tf.saved_model.save(module, "plotproof_acoustic_savedmodel")

def classify(path):
    wav, _ = librosa.load(path, sr=16000, mono=True)
    p = module(wav.astype(np.float32))["probs"].numpy()
    return dict(zip(CLASSES, np.round(p, 3)))

# Sanity only - these are TRAINING clips; the honest numbers are the CV ones above.
print("chainsaw clip:", classify("ESC-50-master/audio/" + meta[meta.category == "chainsaw"].iloc[0].filename))
print("rain clip:    ", classify("ESC-50-master/audio/" + meta[meta.category == "rain"].iloc[0].filename))


In [ ]:
# TFLite for the gateway (laptop / Raspberry Pi). YAMNet's audio frontend needs
# SELECT_TF_OPS, so at inference time load this with the full `tensorflow` pip
# package (tf.lite.Interpreter), not the minimal tflite_runtime.
converter = tf.lite.TFLiteConverter.from_saved_model("plotproof_acoustic_savedmodel")
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS,
                                       tf.lite.OpsSet.SELECT_TF_OPS]
tfl = converter.convert()
open("plotproof_acoustic.tflite", "wb").write(tfl)
print(f"{len(tfl) / 1e6:.1f} MB")

# Verify the TFLite file agrees with the SavedModel on a real clip.
wav, _ = librosa.load("ESC-50-master/audio/" + meta[meta.category == "chainsaw"].iloc[0].filename,
                      sr=16000, mono=True)
it = tf.lite.Interpreter(model_content=tfl)
it.resize_tensor_input(it.get_input_details()[0]["index"], [len(wav)])
it.allocate_tensors()
it.set_tensor(it.get_input_details()[0]["index"], wav.astype(np.float32))
it.invoke()
print("tflite:", dict(zip(CLASSES, np.round(it.get_tensor(it.get_output_details()[0]["index"]), 3))))


In [ ]:
# The model card. Copy acoustic-metrics.json into public/models/ in the
# PlotProof repo - the acoustic page renders it. Every number is the real
# cross-validated measurement from this run; nothing is invented.
from datetime import datetime, timezone
report = classification_report(y_true, y_pred, target_names=CLASSES, digits=3, output_dict=True)
metrics = {
    "modelVersion": "acoustic-yamnet-v1",
    "trainedAt": datetime.now(timezone.utc).isoformat(),
    "task": "5 s clip classification: chainsaw / heavy_vehicle (engine proxy) / other",
    "architecture": "YAMNet (frozen, AudioSet) -> mean+max pool -> 256 dense -> 3-way softmax",
    "dataset": {
        "name": "ESC-50 v2",
        "clips": 2000,
        "license": "CC BY-NC 3.0 - research/demo only; retrain on field audio or FSD50K for production",
    },
    "classes": CLASSES,
    "heavyVehicleProxy": sorted(VEHICLE_PROXY),
    "cv": {
        "protocol": "leave-one-fold-out over the 5 predefined ESC-50 folds",
        "foldAccuracies": [round(a, 4) for a in fold_acc],
        "meanAccuracy": round(float(np.mean(fold_acc)), 4),
        "stdAccuracy": round(float(np.std(fold_acc)), 4),
        "chainsaw": {
            "averagePrecision": round(float(ap), 4),
            "threshold": round(THRESHOLD, 4),
            "precision": round(p_at, 4),
            "recall": round(r_at, 4),
            "support": int(chain_true.sum()),
        },
        "perClass": {c: {"precision": round(report[c]["precision"], 4),
                         "recall": round(report[c]["recall"], 4),
                         "f1": round(report[c]["f1-score"], 4)} for c in CLASSES},
    },
    "tfliteBytes": len(tfl),
    "caveats": [
        "Evaluated on clean 5-second ESC-50 clips; field audio (distance, rain, wind, overlapping sources) is harder and remains unvalidated.",
        "heavy_vehicle is an engine/machinery proxy - ESC-50 has no logging-truck class.",
        "ESC-50 is CC BY-NC: this exact model is for research/demo, not commercial deployment.",
    ],
}
with open("acoustic-metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics["cv"], indent=2))

try:
    from google.colab import files
    files.download("plotproof_acoustic.tflite")
    files.download("acoustic-metrics.json")
except ImportError:
    print("not in Colab - artifacts are in the working directory")


In [ ]:
# Optional: upload any short WAV/MP3 (a YouTube chainsaw clip, your own
# recording) and see what the model says about it.
try:
    from google.colab import files
    up = files.upload()
    for name in up:
        print(name, "->", classify(name))
except ImportError:
    print("Colab-only cell")


## What to do with the artifacts

1. **`acoustic-metrics.json`** → copy into the repo at `public/models/acoustic-metrics.json`
   and deploy. The acoustic page picks it up and renders the model card.
2. **`plotproof_acoustic.tflite`** → keep next to `ml/acoustic/gateway_infer.py` on any
   machine with a microphone (laptop now, Raspberry Pi at a gateway later). Run:

   ```
   pip install tensorflow sounddevice numpy
   set INGEST_URL=https://<your-deployment>/api/acoustic/ingest
   set ACOUSTIC_INGEST_TOKEN=<the bearer token you configured>
   set NODE_LAT=6.75  NODE_LNG=80.303
   python gateway_infer.py
   ```

   It listens in 4-second windows and POSTs real detections to the app's existing
   authenticated ingest endpoint - they appear on the acoustic page and in plot
   evidence packs within seconds.
3. **Retrain honestly when field audio exists:** record at real plots, label,
   drop the clips into the same folder structure, and re-run - the notebook's CV
   protocol and model card generation stay the same.
